# 14 · 从零手撸 Llama-Style Block：RoPE + RMSNorm + SwiGLU + GQA

> **学习目标**：把现代 LLM（Llama / Qwen / DeepSeek 等）的核心 block 拆成 4 个独立组件，**每个单测验证后再组装**。完成后能直接看懂 HuggingFace `modeling_llama.py`。
>
> **预备**：07 + 13 已过（attention + RoPE 基础）。
>
> **为什么重要**：这 4 个组件就是 GPT-2 到 Llama 之间的所有架构差。理解了，你就知道为什么 Llama 比 GPT-2 在同参数下更强。

**架构差异表**：

| 组件 | GPT-2 | Llama (现代) | 收益 |
|------|-------|--------------|------|
| LayerNorm | LayerNorm | **RMSNorm** | 快 ~10%，效果近 |
| FFN | GELU + 2 Linear | **SwiGLU** + 3 Linear | 同 FLOPs 下效果更好 |
| Attention | MHA | **GQA** | KV cache 显存省到 1/8 |
| 位置编码 | 可学习绝对 PE | **RoPE** | 能外推长上下文 |

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import math
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## 1. RMSNorm —— 比 LayerNorm 简洁，效果近

**LayerNorm**：先去均值，再除标准差，再线性变换。`(x - mean) / sqrt(var + eps) * gamma + beta`
**RMSNorm**：**不去均值**，只除「根均方」。`x / sqrt(mean(x²) + eps) * gamma`

**省了 2 件事**：减 mean、加 beta —— 推理 + 训练都更快，效果几乎一致（Llama 论文实验证明）。

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d: int, eps: float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d))   # 只有 gamma，没有 beta
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 沿最后一维算 RMS，再缩放
        rms = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).rsqrt()
        return x * rms * self.weight

# 单测：shape 保持 + 输出 RMS 接近 1（gamma 全 1 时）
x = torch.randn(4, 16, 64) * 3
rn = RMSNorm(64)
y = rn(x)
print('input shape:', x.shape, 'output shape:', y.shape)
print('input  RMS（per pos）:', x.pow(2).mean(-1).sqrt()[0, :3].tolist())
print('output RMS（per pos）:', y.pow(2).mean(-1).sqrt()[0, :3].tolist(), '<- 都 ≈ 1')

# 对比：LayerNorm 还会把均值也归零
ln = nn.LayerNorm(64, elementwise_affine=False)
y_ln = ln(x)
print('\nLayerNorm 输出 mean:', y_ln.mean(-1)[0, :3].tolist(), '<- 都 ≈ 0（去 mean 了）')
print('RMSNorm   输出 mean:', y.mean(-1)[0, :3].tolist(),    '<- *不* 是 0（这是关键区别）')

## 2. SwiGLU —— FFN 升级版

**GPT-2 的 FFN**：`Linear → GELU → Linear`，2 个权重矩阵。

**SwiGLU**：`down( silu(gate(x)) * up(x) )`，3 个权重矩阵。

**直觉**：`silu(gate(x))` 起「门控」作用，决定 `up(x)` 的每个特征通不通过。**门控 = 表达力更强**。

**SiLU**（也叫 Swish）：`x * sigmoid(x)`。0 附近平滑，正半轴近似 ReLU，负半轴有少量负值。

In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, d_model: int, d_ff: int | None = None):
        super().__init__()
        # Llama 默认 d_ff = (2/3) * 4 * d_model，再调整到 64 的倍数
        d_ff = d_ff or int(8 * d_model / 3)
        d_ff = (d_ff + 63) // 64 * 64
        self.gate = nn.Linear(d_model, d_ff, bias=False)
        self.up   = nn.Linear(d_model, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

x = torch.randn(2, 8, 64)
swiglu = SwiGLU(64)
print('SwiGLU 参数量:', sum(p.numel() for p in swiglu.parameters()), f' (3 × Linear)')
print('输出 shape   :', swiglu(x).shape, '应为', (2, 8, 64))

In [ ]:
# 可视化 SiLU vs GELU vs ReLU
xv = torch.linspace(-5, 5, 200)
plt.figure(figsize=(8, 4))
plt.plot(xv, F.relu(xv), label='ReLU')
plt.plot(xv, F.gelu(xv), label='GELU (GPT-2)')
plt.plot(xv, F.silu(xv), label='SiLU / Swish (Llama)', linewidth=2)
plt.axhline(0, color='gray', linewidth=0.5); plt.axvline(0, color='gray', linewidth=0.5)
plt.legend(); plt.grid(True)
plt.title('激活函数对比：ReLU / GELU / SiLU')
plt.show()
print('SiLU 在 0 附近平滑（无 ReLU 的尖角），负半轴有少量负值 —— 配合 SwiGLU 表达力更强。')

## 3. RoPE（复用 13 号思路，整理成可调用模块）

**关键**：cos/sin 表只算一次（属于「buffer」不是参数）。运行时按当前序列长度切片即可。

In [ ]:
def build_rope_cache(seq_len: int, d_head: int, base: float = 10000.0, device='cpu'):
    theta = 1.0 / (base ** (torch.arange(0, d_head, 2, device=device).float() / d_head))
    pos = torch.arange(seq_len, device=device).float()
    freqs = torch.outer(pos, theta)
    return freqs.cos(), freqs.sin()              # 各 (L, d/2)

def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """x: (..., L, d_head)；cos/sin: (L, d_head/2)"""
    x_even, x_odd = x[..., 0::2], x[..., 1::2]
    rot_even = x_even * cos - x_odd * sin
    rot_odd  = x_even * sin + x_odd * cos
    return torch.stack([rot_even, rot_odd], dim=-1).flatten(-2)

# 单测
x = torch.randn(2, 4, 8, 32)        # (B, H, L, d_head)
cos, sin = build_rope_cache(8, 32)
y = apply_rope(x, cos, sin)
print('input shape:', x.shape, '->', 'rope shape:', y.shape)
print('范数应该不变:', x.norm(dim=-1)[0, 0, 0].item(), '->', y.norm(dim=-1)[0, 0, 0].item())

## 4. GQA（Grouped Query Attention）

**MHA（原始）**：H 个 query head，**H 个** key/value head。
**MQA**：H 个 query head，**1 个** key/value head。最省 KV cache，但效果有损失。
**GQA（Llama 2 起）**：H 个 query head，**H/G 个** key/value head（G 组共享）。**折中**，几乎不损质量、显存大省。

**显存收益**：KV cache 大小 ∝ `n_kv_heads`，从 32 head MHA 到 8 head GQA 直接省 75%。

**实现**：把 `Hkv` 个 K/V「复制」成 `H` 个，然后走标准 MHA 计算。

In [ ]:
def repeat_kv(x: torch.Tensor, n_rep: int) -> torch.Tensor:
    """
    x: (B, n_kv_heads, L, d_head)
    重复每个 kv head n_rep 次，使总 head 数 = n_q_heads。
    返回 (B, n_kv_heads * n_rep, L, d_head)
    """
    if n_rep == 1:
        return x
    B, Hkv, L, D = x.shape
    # 在 head 维上插入新轴 -> expand -> reshape
    return x[:, :, None, :, :].expand(B, Hkv, n_rep, L, D).reshape(B, Hkv * n_rep, L, D)

x = torch.arange(8).view(1, 2, 2, 2).float()
print('原 (Hkv=2):', x.flatten().tolist())
print('repeat 3 次 (H=6):', repeat_kv(x, 3).shape, repeat_kv(x, 3)[0, :, 0, 0].tolist())
print('→ 每个 kv head 被复制 3 次连续放在一起')

In [ ]:
class LlamaAttention(nn.Module):
    """GQA + RoPE + causal mask 的 Llama 风格 attention。"""
    def __init__(self, d_model: int, n_heads: int, n_kv_heads: int, max_seq_len: int = 2048):
        super().__init__()
        assert n_heads % n_kv_heads == 0
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.d_model = d_model

        self.wq = nn.Linear(d_model, n_heads    * self.d_head, bias=False)
        self.wk = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.wv = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.wo = nn.Linear(n_heads * self.d_head, d_model, bias=False)

        cos, sin = build_rope_cache(max_seq_len, self.d_head)
        self.register_buffer('rope_cos', cos, persistent=False)
        self.register_buffer('rope_sin', sin, persistent=False)

    def forward(self, x: torch.Tensor):
        B, L, _ = x.shape
        q = self.wq(x).view(B, L, self.n_heads,    self.d_head).transpose(1, 2)  # (B, H,   L, d)
        k = self.wk(x).view(B, L, self.n_kv_heads, self.d_head).transpose(1, 2)  # (B, Hkv, L, d)
        v = self.wv(x).view(B, L, self.n_kv_heads, self.d_head).transpose(1, 2)

        # RoPE 只作用在 q, k（v 不旋）
        cos = self.rope_cos[:L]; sin = self.rope_sin[:L]
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        # GQA：把 K/V 复制到与 Q 同 head 数
        k = repeat_kv(k, self.n_rep)   # (B, H, L, d)
        v = repeat_kv(v, self.n_rep)

        # causal masked attention
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_head)              # (B, H, L, L)
        mask = torch.triu(torch.ones(L, L, device=x.device, dtype=torch.bool), diagonal=1)
        scores = scores.masked_fill(mask, float('-inf'))
        attn = scores.softmax(-1)
        out = attn @ v                                                          # (B, H, L, d)
        out = out.transpose(1, 2).contiguous().view(B, L, self.d_model)
        return self.wo(out)

# Smoke
attn = LlamaAttention(d_model=64, n_heads=8, n_kv_heads=2)
x = torch.randn(2, 16, 64)
y = attn(x)
print('input :', x.shape)
print('output:', y.shape, '应保持 (B, L, D)')

# 参数节省：MHA vs GQA
mha_kv = 64 * 64 * 2                            # K + V each is full D
gqa_kv = 64 * (64 // 8 * 2) * 2                  # K + V 各 = D * (Hkv/H)
print(f'\nKV 参数：MHA={mha_kv}  GQA(2/8)={gqa_kv}  省 {(1-gqa_kv/mha_kv)*100:.0f}%')

## 5. 组装 Llama Block

```
Block:
  h = x + Attention(RMSNorm(x))
  h = h + SwiGLU   (RMSNorm(h))
```

**注意**：和 GPT-2 一样用 Pre-Norm（normalize 在 sub-layer 之前），不同的是 LN → RMSNorm。

In [ ]:
class LlamaBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, n_kv_heads: int, max_seq_len: int = 2048):
        super().__init__()
        self.attn_norm = RMSNorm(d_model)
        self.attn      = LlamaAttention(d_model, n_heads, n_kv_heads, max_seq_len)
        self.ffn_norm  = RMSNorm(d_model)
        self.ffn       = SwiGLU(d_model)

    def forward(self, x):
        x = x + self.attn(self.attn_norm(x))
        x = x + self.ffn(self.ffn_norm(x))
        return x

block = LlamaBlock(d_model=64, n_heads=8, n_kv_heads=2).to(device)
x = torch.randn(2, 16, 64, device=device)
y = block(x)
print('LlamaBlock 输入  :', x.shape)
print('LlamaBlock 输出  :', y.shape, '应保持 (B, L, D)')
print('参数量:', sum(p.numel() for p in block.parameters()), 'fp32')

In [ ]:
# 数值健康检查 1：梯度能流到第一层（没有梯度消失）
x = torch.randn(2, 16, 64, device=device, requires_grad=True)
y = block(x).sum()
y.backward()
grad_norms = {n: p.grad.norm().item() for n, p in block.named_parameters() if p.grad is not None}
print('每层梯度范数（应当全部 > 0）:')
for n, g in list(grad_norms.items())[:6]:
    print(f'  {n:30}  {g:.4f}')
assert all(g > 0 for g in grad_norms.values()), '有梯度 = 0，检查初始化'
print('✅ 梯度流正常')

In [ ]:
# 数值健康检查 2：换位置 → 输出变化（RoPE 的位置敏感性）
torch.manual_seed(0)
block.eval()
x = torch.randn(1, 8, 64, device=device)
y1 = block(x)

# 把第 2 个和第 5 个 token 互换位置
x_swap = x.clone()
x_swap[0, 2], x_swap[0, 5] = x[0, 5].clone(), x[0, 2].clone()
y2 = block(x_swap)

# 与 y1 对应位置的差异
diff = (y1 - y2).abs().max().item()
print(f'交换两个 token 后输出最大差: {diff:.4f}')
print('→ 差应该 >> 0：RoPE 让模型对位置敏感。若 diff ≈ 0 说明 RoPE 没生效。')
assert diff > 0.01

In [ ]:
# 数值健康检查 3：mini batch 训练，loss 单调下降
torch.manual_seed(0)
block_train = LlamaBlock(d_model=64, n_heads=8, n_kv_heads=2).to(device)
head = nn.Linear(64, 64, bias=False).to(device)  # 简单回归头：预测下一个时刻的 mean
opt = torch.optim.AdamW(list(block_train.parameters()) + list(head.parameters()), lr=1e-3)

losses = []
for step in range(50):
    x = torch.randn(8, 16, 64, device=device)
    target = x.roll(-1, dims=1)            # 拿「向右挪 1 位」作为目标（无意义任务，只测能不能拟合）
    pred = head(block_train(x))
    loss = F.mse_loss(pred, target)
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())

import matplotlib.pyplot as plt
plt.figure(figsize=(8, 3))
plt.plot(losses); plt.xlabel('step'); plt.ylabel('loss')
plt.title('LlamaBlock 拟合一个 toy 任务'); plt.grid(True); plt.show()
print(f'最终 loss: {losses[-1]:.4f}（应当 << {losses[0]:.4f}）')

## 深入思考

1. **RMSNorm 比 LayerNorm 真的快 10%？**
   - 在大模型 + GPU 上是的。LayerNorm 要做 mean + var 两个 reduce；RMSNorm 只要 mean(x²) 一个。reduce 在 GPU 上不便宜。
2. **SwiGLU 多了一个 Linear，FLOPs 不是更高？**
   - 是。所以 Llama 把 `d_ff` 从 GPT 的 `4d` 调成 `8d/3`，**总 FLOPs 与 GPT 的 GELU 版相当**。
3. **GQA 多少分组合适？**
   - Llama-2 7B: 32 head / 32 kv（实际是 MHA）。Llama-2 70B: 64 head / 8 kv。Llama-3 8B: 32 head / 8 kv。**经验上 H/Hkv = 4 是甜点**。
4. **为什么 V 不走 RoPE？**
   - RoPE 的本质是「让 Q·K^T 编码相对位置」。V 是「内容向量」，不参与位置匹配，旋了没意义。
5. **RoPE 的 base 改了之后能直接用吗？**
   - 不能。base 决定不同维度对应的频率分布，base 一改 = 模型见到的位置编码完全变了。但**重新训练**就能用更大的 base（Llama-3 用 50w）。

改一改：把 `n_kv_heads` 从 2 改成 8（变成 MHA），看 LlamaBlock 总参数量增加了多少（应主要在 wk、wv）。

## 自检 ✅

- [ ] 不查代码写出 RMSNorm 的 forward 公式。
- [ ] 解释 SwiGLU 比 GELU 多了什么（gate 信号）。
- [ ] 解释 GQA 与 MHA / MQA 的关系。
- [ ] 解释为什么 V 不走 RoPE。
- [ ] 给你 Llama 源码 `modeling_llama.py`，能在 30 分钟内找到 RMSNorm / GQA / RoPE 三段并讲清楚。

## 下一步

→ [`15_kv_cache.ipynb`](15_kv_cache.ipynb)